In [1]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("delta-idempotency")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f1d9aba3-d026-4846-bfd9-eb6fff94e3ad;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [2]:
%%sparksql
CREATE OR REPLACE TABLE default.users (
    id INT,
    name STRING,
    age INT,
    gender STRING,
    country STRING
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/merge-cdc-streaming/users';

In [7]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [8]:
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('country', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))
df = df.withWatermark("ts_ms", "30 seconds")

In [9]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [10]:
def upsertToDelta(microBatchDf, batchId):
    deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/merge-cdc-streaming/users")
    (deltaTable.alias("dt")
    .merge(source=microBatchDf.alias("sdf"), condition="sdf.id = dt.id")
    .whenMatchedUpdate(set={
        "id": "sdf.id",
        "name": "sdf.name",
        "age": "sdf.age",
        "gender": "sdf.gender",
        "country": "sdf.country" })
    .whenNotMatchedInsert(values={
        "id": "sdf.id",
        "name": "sdf.name",
        "age": "sdf.age",
        "gender": "sdf.gender",
        "country": "sdf.country" })
     .execute())
    

In [11]:
query = (df.writeStream
         .format("delta")
         .foreachBatch(upsertToDelta)
         .outputMode("update")
         .option("checkpointLocation", "/opt/workspace/data/delta_lake/merge-cdc-streaming/users/_checkpoints/")
         .trigger(processingTime="5 seconds")
         .start("/opt/workspace/data/delta_lake/merge-cdc-streaming/users"))

In [12]:
%%sparksql

DESCRIBE HISTORY delta.`/opt/workspace/data/delta_lake/merge-cdc-streaming/users`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2025-06-11 07:36:44.391000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(id#406 = id#3556)""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,4,Serializable,False,"{'numOutputRows': '199', 'numTargetBytesAdded': '3162', 'numTargetRowsInserted': '0', 'numTargetFilesAdded': '1', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '1', 'numTargetRowsMatchedUpdated': '4', 'executionTimeMs': '1621', 'numTargetRowsCopied': '195', 'rewriteTimeMs': '371', 'numTargetRowsUpdated': '4', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1114', 'numSourceRows': '1', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '3156'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
4,2025-06-11 07:36:34.305000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(id#406 = id#2781)""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,3,Serializable,False,"{'numOutputRows': '199', 'numTargetBytesAdded': '3156', 'numTargetRowsInserted': '0', 'numTargetFilesAdded': '1', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '1', 'numTargetRowsMatchedUpdated': '1', 'executionTimeMs': '1579', 'numTargetRowsCopied': '198', 'rewriteTimeMs': '368', 'numTargetRowsUpdated': '1', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1102', 'numSourceRows': '1', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '3156'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
3,2025-06-11 07:36:24.548000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(id#406 = id#2006)""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,2,Serializable,False,"{'numOutputRows': '199', 'numTargetBytesAdded': '3156', 'numTargetRowsInserted': '0', 'numTargetFilesAdded': '1', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '1', 'numTargetRowsMatchedUpdated': '1', 'executionTimeMs': '1842', 'numTargetRowsCopied': '198', 'rewriteTimeMs': '514', 'numTargetRowsUpdated': '1', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1237', 'numSourceRows': '1', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '3154'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
2,2025-06-11 07:36:17.145000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(id#406 = id#1231)""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,1,Serializable,False,"{'numOutputRows': '199', 'numTargetBytesAdded': '3154', 'numTargetRowsInserted': '0', 'numTargetFilesAdded': '1', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '1', 'numTargetRowsMatchedUpdated': '2', 'executionTimeMs': '3949', 'numTargetRowsCopied': '197', 'rewriteTimeMs': '1023', 'numTargetRowsUpdated': '2', 'numTargetRowsDeleted': '0', 'scanTimeMs': '2843', 'numSourceRows': '1', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '3148'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
1,2025-06-11 07:36:09.465000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(id#406 = id#436)""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,0,Serializable,False,"{'numOutputRows': '199', 'numTargetBytesAdded': 

25/06/11 07:37:37 ERROR TaskSetManager: Task 0 in stage 135.0 failed 4 times; aborting job
25/06/11 07:39:17 ERROR TaskSetManager: Task 0 in stage 268.0 failed 4 times; aborting job
25/06/11 07:40:57 ERROR TaskSetManager: Task 0 in stage 399.0 failed 4 times; aborting job
25/06/11 07:42:39 ERROR TaskSetManager: Task 0 in stage 532.0 failed 4 times; aborting job
25/06/11 07:44:20 ERROR TaskSetManager: Task 0 in stage 661.0 failed 4 times; aborting job
25/06/11 07:46:00 ERROR TaskSetManager: Task 0 in stage 790.0 failed 4 times; aborting job
25/06/11 07:47:41 ERROR TaskSetManager: Task 0 in stage 917.0 failed 4 times; aborting job
25/06/11 07:49:25 ERROR TaskSetManager: Task 0 in stage 1050.0 failed 4 times; aborting job
25/06/11 07:51:18 ERROR TaskSetManager: Task 0 in stage 1181.0 failed 4 times; aborting job
25/06/11 07:53:21 ERROR TaskSetManager: Task 0 in stage 1312.0 failed 4 times; aborting job
25/06/11 07:55:35 ERROR TaskSetManager: Task 0 in stage 1445.0 failed 4 times; aborting